# 💳 Credit Scoring Model
### CodeAlpha Machine Learning Internship — Task 1

**Objective:** Predict an individual's creditworthiness using past financial data.

**Approach:** Classification algorithms — Logistic Regression, Decision Tree, Random Forest, Gradient Boosting

**Metrics:** Accuracy, Precision, Recall, F1-Score, ROC-AUC

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    precision_score, recall_score, accuracy_score
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 1. Dataset Generation

In [ ]:
np.random.seed(42)
n = 5000

age               = np.random.randint(18, 75, n)
income            = np.random.normal(55000, 25000, n).clip(10000, 200000)
employment_years  = np.random.randint(0, 35, n)
num_credit_lines  = np.random.randint(1, 15, n)
credit_util_ratio = np.random.beta(2, 5, n)
num_late_payments = np.random.poisson(1.2, n)
debt_to_income    = np.random.beta(3, 5, n)
loan_amount       = np.random.normal(15000, 8000, n).clip(1000, 60000)
loan_tenure_yrs   = np.random.randint(1, 10, n)
existing_loans    = np.random.randint(0, 5, n)
savings_balance   = np.random.exponential(8000, n).clip(0, 100000)
has_mortgage      = np.random.binomial(1, 0.4, n)
has_car_loan      = np.random.binomial(1, 0.3, n)

# Creditworthiness score
score = (
    0.30 * (1 - credit_util_ratio)
  + 0.25 * (1 - debt_to_income)
  + 0.20 * (1 - num_late_payments / 10)
  + 0.10 * np.log1p(income) / np.log1p(200000)
  + 0.10 * np.log1p(savings_balance) / np.log1p(100000)
  + 0.05 * employment_years / 35
)
noise = np.random.normal(0, 0.05, n)
creditworthy = ((score + noise) > 0.50).astype(int)

df = pd.DataFrame({
    'age': age, 'income': income.round(2),
    'employment_years': employment_years,
    'num_credit_lines': num_credit_lines,
    'credit_util_ratio': credit_util_ratio.round(4),
    'num_late_payments': num_late_payments,
    'debt_to_income': debt_to_income.round(4),
    'loan_amount': loan_amount.round(2),
    'loan_tenure_yrs': loan_tenure_yrs,
    'existing_loans': existing_loans,
    'savings_balance': savings_balance.round(2),
    'has_mortgage': has_mortgage,
    'has_car_loan': has_car_loan,
    'creditworthy': creditworthy
})

# Introduce realistic missing values
for col in ['income', 'employment_years', 'savings_balance']:
    mask = np.random.rand(n) < 0.02
    df.loc[mask, col] = np.nan

df.to_csv('credit_data.csv', index=False)
print(f'Dataset shape: {df.shape}')
print(f'Class balance: {df.creditworthy.mean():.2%} creditworthy')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print(df.info())
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle('Feature Distributions by Credit Class', fontsize=15, fontweight='bold')
features = ['age','income','employment_years','credit_util_ratio',
            'num_late_payments','debt_to_income','loan_amount','savings_balance','num_credit_lines']
for ax, feat in zip(axes.flat, features):
    for label, color in zip([0,1],['#e74c3c','#2ecc71']):
        ax.hist(df[df.creditworthy==label][feat].dropna(), bins=30, alpha=0.6,
                color=color, label=('Bad' if label==0 else 'Good'))
    ax.set_title(feat.replace('_',' ').title())
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12, 9))
corr = df.select_dtypes(include=np.number).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, square=True)
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering

In [ ]:
df['monthly_income']         = df['income'] / 12
df['monthly_loan_payment']   = df['loan_amount'] / (df['loan_tenure_yrs'] * 12)
df['payment_income_ratio']   = df['monthly_loan_payment'] / df['monthly_income'].replace(0, np.nan)
df['savings_to_income']      = df['savings_balance'] / df['income'].replace(0, np.nan)
df['late_payment_flag']      = (df['num_late_payments'] > 2).astype(int)
df['high_util_flag']         = (df['credit_util_ratio'] > 0.7).astype(int)
df['total_loan_burden']      = df['existing_loans'] + df['has_mortgage'] + df['has_car_loan']
df['age_income_interaction'] = df['age'] * df['income'] / 1e6

print(f'Features after engineering: {df.shape[1]-1}')
print(df.columns.tolist())

## 4. Train / Test Split

In [ ]:
X = df.drop(columns=['creditworthy'])
y = df['creditworthy']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 5. Model Training & Evaluation

In [ ]:
def build_pipeline(clf):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   clf)
    ])

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
}

results = {}
for name, clf in models.items():
    pipe = build_pipeline(clf)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    cv = cross_val_score(pipe, X_train, y_train,
                         cv=StratifiedKFold(5, shuffle=True, random_state=42),
                         scoring='roc_auc')
    results[name] = {
        'pipeline': pipe, 'y_pred': y_pred, 'y_prob': y_prob,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob),
        'cv_auc':    cv.mean()
    }
    print(f"\n{'='*45}\n  {name}\n{'='*45}")
    print(classification_report(y_test, y_pred, target_names=['Bad Credit','Good Credit']))
    print(f'  ROC-AUC: {results[name]["roc_auc"]:.4f}  |  CV AUC: {cv.mean():.4f} ± {cv.std():.4f}')

## 6. Visualisations

In [ ]:
colors = ['#3498db','#e74c3c','#2ecc71','#f39c12']
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC
for (name, r), c in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    axes[0].plot(fpr, tpr, color=c, lw=2, label=f"{name} (AUC={r['roc_auc']:.3f})")
axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_title('ROC Curves', fontweight='bold')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right'); axes[0].grid(alpha=0.3)

# Metric comparison
metrics = ['accuracy','precision','recall','f1','roc_auc']
x = np.arange(len(metrics)); w = 0.18
for i, (name, r) in enumerate(results.items()):
    axes[1].bar(x + i*w, [r[m] for m in metrics], w, label=name, color=colors[i], alpha=0.85)
axes[1].set_xticks(x + w*1.5)
axes[1].set_xticklabels([m.replace('_','\n') for m in metrics])
axes[1].set_ylim(0.5, 1.02); axes[1].set_title('Model Metrics', fontweight='bold')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Bad','Good'], yticklabels=['Bad','Good'])
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
rf = results['Random Forest']['pipeline'].named_steps['model']
fi_df = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_})
fi_df = fi_df.sort_values('importance', ascending=True).tail(15)

plt.figure(figsize=(10, 7))
colors_bar = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(fi_df)))
plt.barh(fi_df['feature'], fi_df['importance'], color=colors_bar)
plt.title('Random Forest — Feature Importances', fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Best Model & Demo Prediction

In [ ]:
best_name = max(results, key=lambda k: results[k]['roc_auc'])
best = results[best_name]

print(f'🏆 Best Model: {best_name}')
print(f'   ROC-AUC   : {best["roc_auc"]:.4f}')
print(f'   Accuracy  : {best["accuracy"]:.4f}')
print(f'   F1-Score  : {best["f1"]:.4f}')

# ── Demo new applicant ──
applicant = pd.DataFrame([{
    'age': 34, 'income': 62000, 'employment_years': 7,
    'num_credit_lines': 4, 'credit_util_ratio': 0.35,
    'num_late_payments': 1, 'debt_to_income': 0.28,
    'loan_amount': 18000, 'loan_tenure_yrs': 5,
    'existing_loans': 1, 'savings_balance': 12000,
    'has_mortgage': 0, 'has_car_loan': 1,
    'monthly_income': 62000/12,
    'monthly_loan_payment': 18000/(5*12),
    'payment_income_ratio': (18000/(5*12))/(62000/12),
    'savings_to_income': 12000/62000,
    'late_payment_flag': 0, 'high_util_flag': 0,
    'total_loan_burden': 2,
    'age_income_interaction': 34*62000/1e6,
}])[X.columns]

prob = best['pipeline'].predict_proba(applicant)[0][1]
pred = best['pipeline'].predict(applicant)[0]

print(f'\n📋 New Applicant Prediction:')
print(f'   Credit Probability : {prob:.2%}')
print(f'   Decision           : {"✅ APPROVED" if pred == 1 else "❌ DECLINED"}')

## Summary

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression | - | - | - | - | - |
| Decision Tree | - | - | - | - | - |
| **Random Forest** | - | - | - | - | **Best** |
| Gradient Boosting | - | - | - | - | - |

**Key Findings:**
- Credit utilization ratio and debt-to-income are the strongest predictors
- Late payment history significantly impacts creditworthiness
- Random Forest / Gradient Boosting outperform simpler models
- Feature engineering improved model performance